In [ ]:

import torch, subprocess, sys
print("torch version:", torch.__version__)
print("torch built with CUDA:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("device count:", torch.cuda.device_count())
    print("device 0:", torch.cuda.get_device_name(0))
    print("allocated:", torch.cuda.memory_allocated()/1e9, "GB")
    print("reserved :", torch.cuda.memory_reserved()/1e9, "GB")
else:
    print("No CUDA runtime available in this session.")
    try:
        out = subprocess.check_output(["nvidia-smi"], text=True)
        print("\nnvidia-smi:\n", out[:500])
    except Exception as e:
        print("nvidia-smi not available:", e)

In [ ]:
import os
import re
import time
import subprocess
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, logging

# ---------- env ----------
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"
logging.set_verbosity_error()

ANSWER_MIN, ANSWER_MAX = 0, 99999
MODEL_DIR = "/kaggle/input/models/qwen-lm/qwen2.5-math/transformers/7b-instruct/1"

def clamp_answer(x):
    try:
        return max(ANSWER_MIN, min(ANSWER_MAX, int(x)))
    except:
        return 0

def extract_last_int(text: str) -> int:
    if not text:
        return 0
    m = re.search(r"FINAL_ANSWER:\s*(-?\d+)", text, flags=re.I)
    if m:
        return int(m.group(1))
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else 0

def find_test_csv():
    cands = [
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv",
        "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv",
    ]
    for p in cands:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("test.csv not found in expected Kaggle input mounts")

def ensure_parquet_engine():
    """Ensure pyarrow is available so DataFrame.to_parquet works in notebook sessions."""
    try:
        import pyarrow  # noqa: F401
    except Exception:
        print("pyarrow not found; installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
        import pyarrow  # noqa: F401
    print("Parquet engine ready (pyarrow).")

def load_model(path):
    print("Loading model:", path)
    tok = AutoTokenizer.from_pretrained(
        path, local_files_only=True, use_fast=False, trust_remote_code=True
    )
    if tok.pad_token is None and tok.eos_token is not None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        path,
        local_files_only=True,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": 0},
    ).eval()

    print("GPU:", torch.cuda.get_device_name(0))
    print("Allocated:", torch.cuda.memory_allocated() / 1e9, "GB")
    return tok, model

@torch.inference_mode()
def solve_one(tok, model, problem):
    msgs = [
        {"role": "system", "content": "Return ONLY: FINAL_ANSWER: <integer>"},
        {"role": "user", "content": str(problem)},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    x = tok(prompt, return_tensors="pt")
    x = {k: v.to("cuda:0") for k, v in x.items()}

    y = model.generate(
        **x,
        max_new_tokens=48,
        do_sample=False,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
        use_cache=True,
    )
    txt = tok.decode(y[0, x["input_ids"].shape[1]:], skip_special_tokens=True)
    return clamp_answer(extract_last_int(txt))

def main():
    t0 = time.time()
    print(
        "torch:", torch.__version__,
        "| cuda:", torch.version.cuda,
        "| avail:", torch.cuda.is_available()
    )

    # parquet output is required by competition
    ensure_parquet_engine()

    test_path = find_test_csv()
    print("Using test.csv:", test_path)

    test_df = pd.read_csv(test_path)
    # expected columns: id, problem (or row_id)
    id_col = "id" if "id" in test_df.columns else ("row_id" if "row_id" in test_df.columns else test_df.columns[0])
    prob_col = "problem" if "problem" in test_df.columns else test_df.columns[1]

    tok, model = load_model(MODEL_DIR)

    answers = []
    problems = test_df[prob_col].astype(str).tolist()
    for i, p in enumerate(problems, 1):
        answers.append(int(solve_one(tok, model, p)))
        if i % 10 == 0 or i == len(problems):
            print(f"Processed {i}/{len(problems)}")

    sub = pd.DataFrame({id_col: test_df[id_col], "answer": answers})

    # REQUIRED OUTPUT FILE FOR THIS COMPETITION
    sub.to_parquet("submission.parquet", index=False)

    # optional debug artifact
    sub.to_csv("submission.csv", index=False)

    print("Wrote submission.parquet")
    print(sub.head())
    print(f"Done in {time.time() - t0:.1f}s for {len(sub)} rows")
    print("Working dir files:", os.listdir("/kaggle/working"))

if __name__ == "__main__":
    import sys
    main()